# Buisness Objective:

Need to get sentiment analysis of financial statements and gauge its impact i.e. positive, negative or neutral on the business and government.


# Data Set

Financial Sentiment Data

# 1.Import Necessary Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np                #library #comprehensive mathematical functions, linear algebra, array, random no etc.
import pandas as pd               #software library #it offers data structures and operations for data manipulation and analysis
import matplotlib.pyplot as plt   #plotting library #pyplot is a collection of functions that make matplotlib work like MATLAB
import seaborn as sns             #data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics.

In [ ]:
#!pip install -U spacy
#!python -m spacy download en
#!pip install wordcloud
#!pip install textblob

In [ ]:
# A list of English positive and negative opinion words or sentiment words taken from the following link
# https://www.cs.uic.edu/~liub/FBS/sentiment-analysis.html

In [ ]:
import spacy      
import re
import nltk
from PIL import Image
from textblob import TextBlob
from nltk.corpus import stopwords
from wordcloud import WordCloud, STOPWORDS

In [ ]:
import nltk
nltk.download('vader_lexicon')
nltk.download('stopwords')
nlp = spacy.load('en_core_web_sm')                             #load the language library

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  #Convert a collection of text documents to a matrix of token counts
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

In [ ]:
#  sklearn : machine learning library
#  feature_extraction : module can be used to extract features in a format supported by machine learning algorithms 
#                       from datasets consisting of formats such as text and image
#  Countvectorizer : it is a method to convert text to numerical data
#  spacy : software library for advanced natural language processing. It features NER, POS tagging, word vectors and more
#  re : This module provides regular expression matching operations. the functions in this module let you check if
#       a particular string matches a given regular expression

# 2. Importing Data Set

In [ ]:
data = pd.read_csv("D:\\PROJECT 2\\financial_sentiment_data.csv")
data

In [ ]:
#showing first five records 
data.head()

# 3.Data Pre-Processing/ Data Understanding

In [ ]:
data.head() #first five records

In [ ]:
data.tail() #last five Records

In [ ]:
data.info() #Basic informtion about data

In [ ]:
data.shape

In [ ]:
data.isna().sum() #cheking null values

In [ ]:
data.describe() #Describe Data

In [ ]:
data.columns #columns names

In [ ]:
data.dtypes  #checking data types of columns

In [ ]:
data.groupby('Sentiment').describe()

In [ ]:
#checking number of characters in each sentence
data['No. of Characters'] = data['Sentence'].str.len()
data.head()

In [ ]:
data = data.drop('No. of Characters',axis=1)

In [ ]:
plt.figure(figsize = (8,6))

sns.histplot(data, x = 'Sentiment', color = 'grey', shrink = 0.9)
plt.title('Number of sentences categorised by Sentiment')

plt.show()

# Text Preprocessing

In [ ]:
#Removing unwanted symbols, punctuations marks etc.

In [ ]:
#creating functions to clean the Sentences

def cleansmt (smt):
  emoji = re.compile("["
         u"\U0001F600-\U0001F64F"  # emoticons
         u"\U0001F300-\U0001F5FF"  # symbols & pictographs
         u"\U0001F680-\U0001F6FF"  # transport & map symbols
         u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
         u"\U00002500-\U00002BEF"  # chinese char
         u"\U00002702-\U000027B0"
         u"\U000024C2-\U0001F251"
         u"\U0001f926-\U0001f937"
         u"\U00010000-\U0010ffff"
         u"\u2640-\u2642" 
         u"\u2600-\u2B55"
         u"\u200d"
         u"\u23cf"
         u"\u23e9"
         u"\u231a"
         u"\ufe0f"                  # dingbats
         u"\u3030"
                      "]+", re.UNICODE)

  smt = re.sub('RT', '', smt)                                    # remove 'RT' from sentence
  smt = re.sub('#[A-Za-z0-9]+', '', smt)                         # remove the '#' from the sentence
  smt = re.sub('\\n', '', smt)                                   # remove the '\n' character
  smt = re.sub('https?:\/\/\S+', '', smt)                        # remove the hyperlinks
  smt = re.sub('@[\S]*', '', smt)                                # remove @mentions
  smt = re.sub('^[\s]+|[\s]+$', '', smt)                         # remove leading and trailing whitespaces
  smt = re.sub(emoji, '', smt)                                   # remove emojis
  smt = re.sub("[^A-Za-z]+"," ",smt).lower()                     # converting to lower
  smt = re.sub("[0-9]+"," ",smt)

  return smt

In [ ]:
#Applying the above function on Data

In [ ]:
data['Sentence'] = data['Sentence'].apply(cleansmt)
data

In [ ]:
#Creating nlp documents for the corpus "Sentence"

In [ ]:
def smt (doc):
    doc = nlp(doc)
    return doc

In [ ]:
#applying above function on data

In [ ]:
data['Sentence'] = data["Sentence"].apply(smt)

In [ ]:
data.head()

# Lemmatization

In [ ]:
for i in range (0,5842):
    for token in data['Sentence'][i]:
        print(f'{token.text:{12}} {token.pos_:{6}} {token.lemma_}')
    print("\n")

In [ ]:
#dataset after lemmatization

lemm_lst=[]
for i in range (0,5842) :
    for token in data['Sentence'][i]:
        lemm_lst.append(token.lemma_)
    lemm_lst.append('..')                                            #making unique identification after end of each sentence

lemm_smt_words = ' '.join(str(e) for e in lemm_lst)                              #joining all lemmatised sentence words
lemm_smt_lst = lemm_smt_words.split("..")                                    #making list of statement after lemmatization
lemm_smt_lst.pop()                                                           #deleting the last element ".."

data['Sentence'] = lemm_smt_lst
data.head()

In [ ]:
#spelling correction

In [ ]:
#data['Sentence'][:100].apply(lambda x: str(TextBlob(x).correct()))               #taking too much time
#data.head()

In [ ]:
#droping empty rows

In [ ]:
data.drop(data[data['Sentence'] == ''].index, inplace = True)
data.reset_index(drop=True, inplace=True)

In [ ]:
#findings duplicates

In [ ]:
data[data.duplicated()].shape                       #checking no. of duplicate records

In [ ]:
data[data.duplicated()]                             #finding particular duplicate records 

In [ ]:
data[data.duplicated(keep = False)].shape

In [ ]:
data[data.duplicated(keep = False)]

In [ ]:
#droping duplicates

data=data.drop_duplicates()
data.shape

In [ ]:
#handling partial duplicates (checking duplicates in Sentence column only)

In [ ]:
data['Sentence'][data['Sentence'].duplicated()].shape

In [ ]:
data_dup = data[data['Sentence'].duplicated()]
data_dup

In [ ]:
data_falsedup = data[data['Sentence'].duplicated(keep = False)] 
data_falsedup

In [ ]:
data_falsedup['Sentiment'].value_counts()

In [ ]:
# let's find out information from partial duplicate records using wordcloud

In [ ]:
def convert(lst):
    return ''.join(lst).split()

 
dup_lst =  convert(data_dup.Sentence)

In [ ]:
#separating positive words

with open("D:\\PROJECT 2\\positive-words.txt","r") as pos:
    poswords = pos.read().split("\n")
    
#positive words wordcloud
pos_words = " ".join([w for w in dup_lst if w in poswords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(pos_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

print(len(pos_words))
print(pos_words)

In [ ]:
#separating negative words

with open("D:\\PROJECT 2\\negative-words.txt","r") as pos:
    negwords = pos.read().split("\n")
    
#positive words wordcloud
neg_words = " ".join([w for w in dup_lst if w in negwords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(neg_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

print(len(neg_words))
neg_words

In [ ]:
# # majority of words in duplicated records are negative (1704) and only 389 words are positive
# so we can say, out of 507 records most of the records might have negative sentiment

# now we will analyse sentiment of duplicated records by using polarity score concept

In [ ]:
# let's find out information from partial duplicated records using polarity score

In [ ]:
data_falsedup['p_score'] = data_falsedup['Sentence'].apply(lambda x : sia.polarity_scores(x))
data_falsedup['c_score'] = data_falsedup['p_score'].apply(lambda scores: scores['compound'])
data_falsedup.head()

In [ ]:
def get_sentiment(value):
    if value < 0:
        return 'negative'
    elif value > 0:
        return 'positive'
    else:
        return 'neutral'
    
data_falsedup['Sentiment'] = data_falsedup['c_score'].apply(get_sentiment)
data_falsedup.head()

In [ ]:
data_falsedup['Sentiment'].value_counts()

In [ ]:
del data_falsedup['p_score']
del data_falsedup['c_score']

In [ ]:
data_falsedup = data_falsedup.drop_duplicates()
data_falsedup.shape

In [ ]:
# droping the partial duplicates and concating original data free from partial duplicates and data_falsedup after performing polarity

In [ ]:
data=data.drop_duplicates(subset=['Sentence'],keep = False)
data.shape

In [ ]:
data = pd.concat([data, data_falsedup], axis=0)
data.shape

# Removing stop words

In [ ]:
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

In [ ]:
stop = nltk.download('stopwords')  # Downloading stop words
stop = set(stopwords.words('english'))  # Selecting English stop words
data['Sentence'] = data['Sentence'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
data.head()

# 4. Plotting

## a. Bar Plot

In [ ]:
import matplotlib.pyplot as plt
#bar plot
plt.Figure(figsize=(14,16))
t1=(data['Sentiment']).value_counts()
t1
t1.plot(kind="bar")


## b. Pie chart

In [ ]:
plt.figure(figsize=(5,5))
data.Sentiment.value_counts().plot(kind='pie', labels=['positive', 'neutral', 'negative'],
                              wedgeprops=dict(width=.7), autopct='%1.0f%%', startangle= -20, 
                              textprops={'fontsize': 15})

In [ ]:
#pie Chart
plt.title("Sentiment")
t1.plot(kind='pie')

# 5.Wordcloud

In [ ]:
grouped = data.groupby(data.Sentiment)
data_pos = grouped.get_group("positive")
data_neg = grouped.get_group("negative")
data_neu = grouped.get_group("neutral")

In [ ]:
def convert(lst):
    return ''.join(lst).split()

 
pos_lst =  convert(data_pos.Sentence)
neg_lst =  convert(data_neg.Sentence)
neu_lst =  convert(data_neu.Sentence)
#pos_lst

In [ ]:
#wordcloud for positive sentiment

wc_pos = " ".join([w for w in pos_lst])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(wc_pos)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(cloud.words_.keys())

In [ ]:
from collections import Counter
word_set = wc_pos
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#wordcloud for negative sentiment

wc_neg = " ".join([w for w in neg_lst])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(wc_neg)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(cloud.words_.keys())

In [ ]:
from collections import Counter
word_set = wc_neg
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#wordcloud for neutral sentiment

wc_neu = " ".join([w for w in neu_lst])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(wc_neu)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(cloud.words_.keys())

In [ ]:
from collections import Counter
word_set = wc_neu
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#positive words wordcloud for positive sentiment

In [ ]:
#separating positive words

with open("D:\\PROJECT 2\\positive-words.txt","r") as pos:
    poswords = pos.read().split("\n")
    
#positive words wordcloud
pos_words = " ".join([w for w in pos_lst if w in poswords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(pos_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(pos_words))
pos_words

In [ ]:
print(cloud.words_.keys())

In [ ]:
from collections import Counter
word_set = pos_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#negative words wordcloud for positive sentiment

In [ ]:
with open("D:\\PROJECT 2\\negative-words.txt","r") as pos:
    negwords = pos.read().split("\n")
    
#negative word cloud
neg_words = " ".join([w for w in pos_lst if w in negwords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 200, stopwords = set(STOPWORDS))
cloud

cloud.generate(neg_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(neg_words))
neg_words

In [ ]:
from collections import Counter
word_set = neg_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#  positive words wordcloud for negative sentiment

In [ ]:
#separating positive words

with open("D:\\PROJECT 2\\positive-words.txt","r") as pos:
    poswords = pos.read().split("\n")
    
#positive words wordcloud
pos_words = " ".join([w for w in neg_lst if w in poswords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(pos_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(pos_words))
pos_words

In [ ]:
from collections import Counter
word_set = pos_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
# negative words wordcloud for negative sentiment

In [ ]:
with open("D:\\PROJECT 2\\negative-words.txt","r") as pos:
    negwords = pos.read().split("\n")
    
#negative word cloud
neg_words = " ".join([w for w in neg_lst if w in negwords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 200, stopwords = set(STOPWORDS))
cloud

cloud.generate(neg_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(neg_words))
neg_words

In [ ]:
from collections import Counter
word_set = neg_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#positive words wordcloud for neutral sentiment

In [ ]:
#separating positive words

with open("D:\\PROJECT 2\\positive-words.txt","r") as pos:
    poswords = pos.read().split("\n")
    
#positive words wordcloud
pos_words = " ".join([w for w in neu_lst if w in poswords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 300, stopwords = set(STOPWORDS))
cloud

cloud.generate(pos_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(pos_words))
pos_words

In [ ]:
from collections import Counter
word_set = pos_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

In [ ]:
#negative words wordcloud for neutral sentiment

In [ ]:
with open("D:\\PROJECT 2\\negative-words.txt","r") as pos:
    negwords = pos.read().split("\n")
    
#negative word cloud
neg_words = " ".join([w for w in neu_lst if w in negwords])

#maskarray = np.array(Image.open("butterfly.png"))   mask = maskarray,
cloud = WordCloud(background_color = "black", max_words = 200, stopwords = set(STOPWORDS))
cloud

cloud.generate(neg_words)
#cloud.to_file("wordcloud.png")
plt.axis("off")
plt.imshow(cloud)

In [ ]:
print(len(neg_words))
neg_words

In [ ]:
from collections import Counter
word_set = neg_words
word_list = word_set.split()

word_count = Counter(word_list)

print(word_count.most_common(50))

# 6.Feature Extraction

In [ ]:
# #export dataframe to excel
# from sklearn.feature_extraction.text import CountVectorizer
# DataFrame_name = pd.DataFrame(CountVectorizer.get_feature_names_out(self=))

# #pip install openpyxl
# writer = pd.ExcelWriter('converted-to-excel.xlsx')
# DataFrame_name.to_excel(writer)
# writer.save()

# Count Vectorizer(Unigram)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1,1))         #(1,1) means unigram; (2,2) means bigram; (1,2) means unigram + bigram
x1 = vectorizer.fit_transform(data['Sentence'])

In [ ]:
print(vectorizer.get_feature_names_out()[0:100])

In [ ]:
x1.shape

In [ ]:
pd.DataFrame(x1.toarray()) 

In [ ]:
def plot_top_ngrams_barchart(text, n=1):
    stop=set(stopwords.words('english'))

    new= text.str.split()
    new=new.values.tolist()
    corpus=[word for i in new for word in i]

    def _get_top_ngram(corpus, n=None):
        vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
        bag_of_words = vec.transform(corpus)
        sum_words = bag_of_words.sum(axis=0) 
        words_freq = [(word, sum_words[0, idx]) 
                      for word, idx in vec.vocabulary_.items()]
        words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
        return words_freq[:10]

    top_n_bigrams=_get_top_ngram(text,n)[:10]
    x,y=map(list,zip(*top_n_bigrams))
    sns.barplot(x=y,y=x)

In [ ]:
plot_top_ngrams_barchart(data['Sentence'],1)

plt.xlabel('Number of Non - Stopwords')
plt.ylabel('uni - gram')
plt.show()

# count vectorizer(bigram)

In [ ]:
# #export dataframe to excel

# #pip install openpyxl
# writer = pd.ExcelWriter('converted-to-excel.xlsx')
# DataFrame_name.to_excel(writer)
# writer.save()

In [ ]:
vectorizer = CountVectorizer(ngram_range=(2,2))         #(1,1) means unigram; (2,2) means bigram; (1,2) means unigram + bigram
x2 = vectorizer.fit_transform(data['Sentence'])

In [ ]:
x2.shape

In [ ]:
pd.DataFrame(x2.toarray())

In [ ]:
def plot_top_ngrams_barchart(text, n=2):
    stop=set(stopwords.words('english'))

    new= text.str.split()
    new=new.values.tolist()
    corpus=[word for i in new for word in i]

    def _get_top_ngram(corpus, n=None):
        vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
        bag_of_words = vec.transform(corpus)
        sum_words = bag_of_words.sum(axis=0) 
        words_freq = [(word, sum_words[0, idx]) 
                      for word, idx in vec.vocabulary_.items()]
        words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
        return words_freq[:10]

    top_n_bigrams=_get_top_ngram(text,n)[:10]
    x,y=map(list,zip(*top_n_bigrams))
    sns.barplot(x=y,y=x)

In [ ]:
plot_top_ngrams_barchart(data['Sentence'],2)

plt.xlabel('Number of Non - Stopwords')
plt.ylabel('bi - gram')
plt.show()

# count vectorizer(Trigram)

In [ ]:
vectorizer = CountVectorizer(ngram_range=(3,3))         #(1,1) means unigram; (2,2) means bigram; (1,2) means unigram + bigram
x3 = vectorizer.fit_transform(data['Sentence'])

In [ ]:
x3.shape

In [ ]:
pd.DataFrame(x3.toarray())

In [ ]:
def plot_top_ngrams_barchart(text, n=3):
    stop=set(stopwords.words('english'))

    new= text.str.split()
    new=new.values.tolist()
    corpus=[word for i in new for word in i]

    def _get_top_ngram(corpus, n=None):
        vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
        bag_of_words = vec.transform(corpus)
        sum_words = bag_of_words.sum(axis=0) 
        words_freq = [(word, sum_words[0, idx]) 
                      for word, idx in vec.vocabulary_.items()]
        words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
        return words_freq[:10]

    top_n_bigrams=_get_top_ngram(text,n)[:10]
    x,y=map(list,zip(*top_n_bigrams))
    sns.barplot(x=y,y=x)

In [ ]:
plot_top_ngrams_barchart(data['Sentence'],3)

plt.xlabel('Number of Non - Stopwords')
plt.ylabel('tri - gram')
plt.show()

# CountVectorizer(Unigram+Trigram)

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1,3))         #(1,1) means unigram; (2,2) means bigram; (1,2) means unigram + bigram
x4 = vectorizer.fit_transform(data['Sentence'])

In [ ]:
x4.shape

In [ ]:
pd.DataFrame(x4.toarray())

# TF-IDF unigram

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform(data['Sentence'])

In [ ]:
tfidf.shape

In [ ]:
pd.DataFrame(tfidf.toarray())

# TF-IDF bigram

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(2,2))

tfidf = vectorizer.fit_transform(data['Sentence'])

In [ ]:
tfidf.shape

In [ ]:
pd.DataFrame(tfidf.toarray())

# TfIDF Trigram

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(3,3))

tfidf = vectorizer.fit_transform(data['Sentence'])

In [ ]:
tfidf.shape

In [ ]:
pd.DataFrame(tfidf.toarray())

# TFIDF(Unigram+Trigram)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,3))

tfidf = vectorizer.fit_transform(data['Sentence'])

In [ ]:
tfidf.shape

In [ ]:
pd.DataFrame(tfidf.toarray())

# 7.Model Building

In [ ]:
data

# 7.1 Using CountVectorizer

In [ ]:
#Creating Bag of Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold, cross_val_score, train_test_split

In [ ]:
cv= CountVectorizer(ngram_range=(1, 3), max_features=10000)
X = cv.fit_transform(data['Sentence']).toarray()

In [ ]:
data['Sentiment'].replace({'positive':1,'negative':-1,'neutral':0},inplace=True)

In [ ]:
y = data['Sentiment']
y

In [ ]:
#Splitting Data Set in Train and Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state=0)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

# 7.1.1 Gaussian Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
GNB = GaussianNB()
GNB.fit(X_train, y_train)

In [ ]:
y_pred_GNB = GNB.predict(X_test)
y_pred_GNB

In [ ]:
#Accuracy of GaussianNB Algorithm
Acc_GNB = accuracy_score(y_pred_GNB, y_test)
print('Accuracy of GaussianNB :', Acc_GNB)

In [ ]:
#Classification Report
report = classification_report(y_pred_GNB, y_test)
print(report)

In [ ]:
#Making the Confusion Matrix

In [ ]:
CM_GNB = confusion_matrix(y_test, y_pred_GNB) 
CM_GNB

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_GNB, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for GaussianNB Model")
plt.show()

# 7.1.2 SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
classifier = SVC()

In [ ]:
classifier.fit(X_train,y_train)

In [ ]:
y_pred  = classifier.predict(X_test)

In [ ]:
y_pred

In [ ]:
#Cheking accuracy
svm_ac = accuracy_score(y_pred, y_test)
print('Accuracy of svm :', svm_ac)

In [ ]:
report = classification_report(y_pred, y_test)
print(report)

In [ ]:
CM_SVM = confusion_matrix(y_test, y_pred) 
CM_SVM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_SVM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for SVM ")
plt.show()

# 7.1.3 LightGBM Classifier Model

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
LGBM = LGBMClassifier()
LGBM.fit(X_train, y_train)

In [ ]:
y_pred_LGBM = LGBM.predict(X_test)
y_pred_LGBM

In [ ]:
Acc_LGBM = accuracy_score(y_pred_LGBM, y_test)
print('Accuracy of LightGBM Classifier :', Acc_LGBM)

In [ ]:
report = classification_report(y_pred_LGBM, y_test)
print(report)

In [ ]:
#Making the Confusion Matrix

In [ ]:
CM_LGBM = confusion_matrix(y_test, y_pred_LGBM) 
CM_LGBM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_LGBM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for LGBM Model")
plt.show()

# 7.1.4 Random Forest Classifier Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RFC = RandomForestClassifier(criterion = 'entropy', random_state = 0)
RFC.fit(X_train, y_train)

In [ ]:
y_pred_RFC = RFC.predict(X_test)
y_pred_RFC

In [ ]:
Acc_RFC = accuracy_score(y_pred_RFC, y_test)
print('Accuracy of Random Forest Classifier :', Acc_RFC)

In [ ]:
report = classification_report(y_pred_RFC, y_test)
print(report)

In [ ]:
CM_RFC = confusion_matrix(y_test, y_pred_RFC) 
CM_RFC

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_RFC, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for Random Forest Model")
plt.show()

In [ ]:
list = [['Gaussian Naive Bayes',Acc_GNB],['Support Vector Machine',svm_ac],
        ['LightGBM Classifier ',Acc_LGBM],['Random Forest Classifier',Acc_RFC]]

In [ ]:
Result = pd.DataFrame(list, columns = ['Models', 'Accuracy']) 
Result

In [ ]:
plt.figure(figsize = (15,6))
sns.barplot(data = Result,x = 'Models',y = 'Accuracy')

plt.title('Accuracy vs Model Plot', fontsize=20)
plt.xlabel('Models', fontsize=18)
plt.ylabel('Accuracy', fontsize=18)

plt.xticks(rotation = 90)
plt.show()

# 7.2 Using TF-IDF

In [ ]:
#Creating Bag of Words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X = tfidf.fit_transform(data['Sentence']).toarray()

In [ ]:
data['Sentiment'].replace({'positive':1,'negative':-1,'neutral':0},inplace=True)

In [ ]:
y = data['Sentiment']

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state=0)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

# 7.2.1 Gaussian Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
GNB = GaussianNB()
GNB.fit(X_train, y_train)

In [ ]:
y_pred_GNB = GNB.predict(X_test)
y_pred_GNB

In [ ]:
Acc_GNB = accuracy_score(y_pred_GNB, y_test)
print('Accuracy of GaussianNB :', Acc_GNB)

In [ ]:
report = classification_report(y_pred_GNB, y_test)
print(report)

In [ ]:
CM_GNB = confusion_matrix(y_test, y_pred_GNB) 
CM_GNB

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_GNB, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for GaussianNB Model")
plt.show()

# 7.2.2 SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
classifier = SVC()

In [ ]:
classifier.fit(X_train,y_train)

In [ ]:
y_pred  = classifier.predict(X_test)

In [ ]:
y_pred

In [ ]:
#Cheking accuracy
svm_ac = accuracy_score(y_pred, y_test)
print('Accuracy of svm :', svm_ac)

In [ ]:
report = classification_report(y_pred, y_test)
print(report)

In [ ]:
CM_SVM = confusion_matrix(y_test, y_pred) 
CM_SVM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_SVM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for SVM ")
plt.show()

# 7.2.3 LightGBM Classifier Model

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
LGBM = LGBMClassifier()
LGBM.fit(X_train, y_train)

In [ ]:
y_pred_LGBM = LGBM.predict(X_test)
y_pred_LGBM

In [ ]:
Acc_LGBM = accuracy_score(y_pred_LGBM, y_test)
print('Accuracy of LightGBM Classifier :', Acc_LGBM)

In [ ]:
report = classification_report(y_pred_LGBM, y_test)
print(report)

In [ ]:
CM_LGBM = confusion_matrix(y_test, y_pred_LGBM) 
CM_LGBM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_LGBM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for LGBM Model")
plt.show()

# 7.2.4 Random Forest Classifier Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RFC = RandomForestClassifier(criterion = 'entropy', random_state = 0)
RFC.fit(X_train, y_train)

In [ ]:
y_pred_RFC = RFC.predict(X_test)
y_pred_RFC

In [ ]:
Acc_RFC = accuracy_score(y_pred_RFC, y_test)
print('Accuracy of Random Forest Classifier :', Acc_RFC)

In [ ]:
report = classification_report(y_pred_RFC, y_test)
print(report)

In [ ]:
CM_RFC = confusion_matrix(y_test, y_pred_RFC) 
CM_RFC

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_RFC, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for Random Forest Model")
plt.show()

In [ ]:
list = [['Gaussian Naive Bayes',Acc_GNB],['Support Vector Machine',svm_ac],
        ['LightGBM Classifier ',Acc_LGBM],['Random Forest Classifier',Acc_RFC]]

In [ ]:
Result = pd.DataFrame(list, columns = ['Models', 'Accuracy']) 
Result

In [ ]:
plt.figure(figsize = (15,6))
sns.barplot(data = Result,x = 'Models',y = 'Accuracy')

plt.title('Accuracy vs Model Plot', fontsize=20)
plt.xlabel('Models', fontsize=18)
plt.ylabel('Accuracy', fontsize=18)

plt.xticks(rotation = 90)
plt.show()

# 7.3 Using Word2Vec

In [ ]:
!pip install gensim==4.2.0


In [ ]:
import gensim, logging
from gensim.models import word2vec
from gensim.models.word2vec import Word2Vec
import string
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
#using pretrianed word vectors

In [ ]:
import gensim.downloader as api
print(list(gensim.downloader.info()['models'].keys()))

In [ ]:
wv = api.load('glove-twitter-25')

In [ ]:
type(wv)

In [ ]:
wv['fall']

In [ ]:
len(wv['fall'])

In [ ]:
#finding most similar words to fall (fall is available in out corpus)
wv.most_similar(positive="fall")

In [ ]:
data.head()

In [ ]:
#converting each sentence into vectors

In [ ]:
def smt_vec(smt):
    vector_size = wv.vector_size
    wv_res = np.zeros(vector_size)
    ctr = 1
    for w in smt:
        if w in wv:
            ctr += 1
            wv_res += wv[w]
    wv_res = wv_res/ctr
    return wv_res

In [ ]:
smt_vec('i am happy')

In [ ]:
data1=data

In [ ]:
data1['tokenized_sentence'] = data1['Sentence'].apply(lambda x: x.split()) # tokenizing 
data1['tokenized_vector'] = data1['tokenized_sentence'].apply(smt_vec)
data1.head()

In [ ]:
data1['tokenized_vector'][0]

In [ ]:
x = data['tokenized_vector'].to_list()
y = data['Sentiment'].to_list()

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify = y)

# 7.3.1 Gaussian Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
GNB = GaussianNB()
GNB.fit(x_train, y_train)

In [ ]:
y_pred_GNB = GNB.predict(x_test)
y_pred_GNB

In [ ]:
Acc_GNB = accuracy_score(y_pred_GNB, y_test)
print('Accuracy of GaussianNB :', Acc_GNB)

In [ ]:
report = classification_report(y_pred_GNB, y_test)
print(report)

In [ ]:
CM_GNB = confusion_matrix(y_test, y_pred_GNB) 
CM_GNB

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_GNB, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for GaussianNB Model")
plt.show()

# 7.3.2 SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
classifier = SVC()

In [ ]:
classifier.fit(x_train,y_train)

In [ ]:
y_pred  = classifier.predict(x_test)

In [ ]:
y_pred

In [ ]:
#Cheking accuracy
svm_ac = accuracy_score(y_pred, y_test)
print('Accuracy of svm :', svm_ac)

In [ ]:
report = classification_report(y_pred, y_test)
print(report)

In [ ]:
CM_SVM = confusion_matrix(y_test, y_pred) 
CM_SVM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_SVM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for SVM ")
plt.show()

# 7.3.3 LightGBM Classifier Model

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
LGBM = LGBMClassifier()
LGBM.fit(x_train, y_train)

In [ ]:
y_pred_LGBM = LGBM.predict(x_test)
y_pred_LGBM

In [ ]:
Acc_LGBM = accuracy_score(y_pred_LGBM, y_test)
print('Accuracy of LightGBM Classifier :', Acc_LGBM)

In [ ]:
report = classification_report(y_pred_LGBM, y_test)
print(report)

In [ ]:
CM_LGBM = confusion_matrix(y_test, y_pred_LGBM) 
CM_LGBM

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_LGBM, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for LGBM Model")
plt.show()

# 7.3.4 Random Forest Classifier Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RFC = RandomForestClassifier(criterion = 'entropy', random_state = 0)
RFC.fit(x_train, y_train)

In [ ]:
y_pred_RFC = RFC.predict(x_test)
y_pred_RFC

In [ ]:
Acc_RFC = accuracy_score(y_pred_RFC, y_test)
print('Accuracy of Random Forest Classifier :', Acc_RFC)

In [ ]:
report = classification_report(y_pred_RFC, y_test)
print(report)

In [ ]:
CM_RFC = confusion_matrix(y_test, y_pred_RFC) 
CM_RFC

In [ ]:
plt.figure(figsize=(4,3))

sns.heatmap(CM_RFC, annot=True, fmt = ".1f",cmap="RdBu")
plt.title("Confusion Matrix for Random Forest Model")
plt.show()

In [ ]:
list = [['Gaussian Naive Bayes',Acc_GNB],['Support Vector Machine',svm_ac],
        ['LightGBM Classifier ',Acc_LGBM],['Random Forest Classifier',Acc_RFC]]

In [ ]:
Result = pd.DataFrame(list, columns = ['Models', 'Accuracy']) 
Result

In [ ]:
plt.figure(figsize = (15,6))
sns.barplot(data = Result,x = 'Models',y = 'Accuracy')

plt.title('Accuracy vs Model Plot', fontsize=20)
plt.xlabel('Models', fontsize=18)
plt.ylabel('Accuracy', fontsize=18)

plt.xticks(rotation = 90)
plt.show()

# 7.4 MODEL SAVING

In [ ]:
import pickle

In [ ]:
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))

In [ ]:
pickle.dump(RFC, open('RFC.pkl', 'wb'))

In [ ]:
pickled_model = pickle.load(open('RFC.pkl', 'rb'))
pickled_model.predict(X_test)

In [ ]:
# saving data into csv for model buiding pov
data.to_csv('data.csv')